In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import importlib

from kfirlib.comparisons import *
from kfirlib.util import *
from kfirlib import pubmed, gpt

In [ ]:
DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables\fromJamalah9_6'

In [ ]:
orig_df = pd.read_excel(os.path.join(DIR, '26_with_inclusions.xlsx'))
orig_df.head()

## Find in Pubmed

In [ ]:
BATCH_SIZE = 5

In [ ]:
results = []
for title_batch in tqdm(list(batched(orig_df.Title, BATCH_SIZE))):
    query = ' OR '.join(f'"{title}"' for title in title_batch)
    res_df = pubmed.query(query, max_results=3 * BATCH_SIZE)
    results.append((title_batch, res_df))

In [ ]:
good_parts = []
bad_parts = []

for titles, df in results:
    query_extracts = {extract_letters(title) for title in titles}
    df['title_extract'] = df['title'].apply(extract_letters)
    df['match'] = df.title_extract.isin(query_extracts)

    if df.match.any():
        good_parts.append(df[df.match])
    not_found = query_extracts - set(df.title_extract)
    if not_found:
        bad_parts.append((not_found, df[~df.match]))
        

In [ ]:
sum(len(df) for df in good_parts)

In [ ]:
len(bad_parts)

In [ ]:
sum(len(p[0]) for p in bad_parts)

In [ ]:
ind = 3
bad = bad_parts[ind]
print(bad[0])
bad[1].title

In [ ]:
found_df = pd.concat(good_parts, axis=0)
assert(found_df.title_extract.nunique() == len(found_df.title_extract))
orig_df['title_extract'] = orig_df.Title.apply(extract_letters)
filled_df = pd.merge(orig_df, found_df, on='title_extract', how='left')
filled_df.head()

In [ ]:
filled_df.columns

In [ ]:
filled_df.isna().sum()

In [ ]:
filled_df.drop(columns=['title', 'authors', 'match'], inplace=True)

In [ ]:
filled_df['abstract'] = filled_df['abstract_x'].fillna(filled_df['abstract_y'])
filled_df['abstract'].isna().sum()

In [ ]:
filled_df.drop(columns=['abstract_x', 'abstract_y'], inplace=True)

In [ ]:
filled_df.to_excel(os.path.join(DIR, '26_with_pubmed.xlsx'), index=False)

## GPT

In [ ]:
upd_df = pd.read_excel(os.path.join(DIR, '26_with_pubmed.xlsx'))

In [ ]:
upd_df.columns

In [ ]:
del upd_df['Unnamed: 0']

In [ ]:
upd_df.rename(columns={'Year ': 'Year'}, inplace=True)

In [ ]:
eligibility_df = gpt.check_eligibility(upd_df, criteria=gpt.JUMALAH_CRITERIA)

In [ ]:
eligibility_df.decision.sum()

In [ ]:
eligibility_df.to_excel(os.path.join(DIR, '26_with_decision.xlsx'), index=False)